In [1]:
import config as cfg
from data_acquirer import get_data, gen_data
from data_processor import process
import helpers
import heuristic
from heuristic import Path
from solver import solve_pdpcr


import copy
import datetime
from docplex.mp.model import Model
import heapq
import numpy as np
import time


In [ ]:
method = 5
data_name = 'generated3.csv' #real data

if False:
    is_depot_task = True
else:
    is_depot_task = False #TODO remove this

    raw = get_data(data_name,'csv', is_depot_task)

In [6]:

#WR2 data
sec_names = ['100','200','400','600','800','1000']
third_names = ['1','2','3','4','5']


data_paths =[f'wr2/bar-n{j}-{k}-3.csv'for j in sec_names for k in third_names]#TODO add data
dist_paths = [f'wr2/bar-n{j}-{k}-dist.csv'for j in sec_names for k in third_names]#TODO add data
num_robots = [5*i for i in range(2,3)]

method = 5
file_type = 'csv'
is_depot_task = False


In [12]:

#Li larger data

first_names = ['LC1']
sec_names = [2,4,6,8,10]
third_names = [i for i in range(1,11)]
data_paths =[f'lim/{j}00/{i}_{j}_{k}-5-uniform.csv'for i in first_names for j in sec_names for k in third_names]#
num_robots = [10*i for i in range(1,4)]
method = 5
file_type = 'csv'
is_depot_task = False

In [ ]:
# CPLEX data li lim
for data_path in data_paths:
    for num_robot in num_robots:
        try:
            print(f'Solving for {data_path} with {num_robot} robots')
            raw = get_data(data_path, file_type, is_depot_task)
            processed_data = process(raw, method)
            solve_result = solve_pdpcr(processed_data, num_robot, method)
            
            solve_log = []
            solve_log.append(data_path)
            solve_log.append(num_robot)
            solve_log.append(solve_result['solve_time'])
            solve_log.append(solve_result['sol'].objective_value)
            solve_log.append(solve_result['processed_nodes'])
            solve_log.append(solve_result['gap'])
            now = datetime.datetime.now()
            solve_log.append(now.strftime("%Y-%m-%d %H:%M:%S"))

            with open('solve_log.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)

In [ ]:
## Mass solve using heuristics
for data_path in data_paths:
    for num_robot in num_robots:
        try:
            print(f'Solving for {data_path} with {num_robot} robots')
            raw = get_data(data_path, file_type, is_depot_task)
            processed_data = process(raw, method)
            processed_h = heuristic.process_greedy_1(raw)
            tik1 = time.time()
            solve_result_h = heuristic.solve_greedy_1(processed_h, num_robot)
            tok1 = time.time()
            greedy1_time = tok1 - tik1
            print(f'Greedy 1 time: {greedy1_time}')
            tik2 = time.time()
            solve_result_h_jp = heuristic.solve_greedy_jp(processed_h, num_robot)
            tok2 = time.time()
            greedy_jp_time = tok2 -tik2
            print(f'Greedy jp time: {greedy_jp_time}')
            tik3 = time.time()
            solve_result_h_jp_2 = heuristic.solve_greedy_jp_compare_start_time(processed_h, num_robot)
            tok3 = time.time()
            greedy_jp_time_2 = tok3 -tik3
            print(f'Greedy jp time 2: {greedy_jp_time_2}')

        
            solve_log = []
            solve_log.append(data_path)
            solve_log.append(solve_result_h.get_makespan().round(2))
            solve_log.append(np.round(greedy1_time,2))
            solve_log.append(solve_result_h_jp.get_makespan().round(2))
            solve_log.append(np.round(greedy_jp_time,2))
            solve_log.append(solve_result_h_jp_2.get_makespan().round(2))
            solve_log.append(np.round(greedy_jp_time_2,2))
            with open('solve_log_h_2.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)

In [ ]:
import helpers
## Mass solve 2 opt
for data_path in data_paths:
    for num_robot in num_robots:
        try:
            print(f'Solving for {data_path} with {num_robot} robots')
            raw = get_data(data_path, file_type, is_depot_task)
            processed_h = heuristic.process_greedy_1(raw)

            tik2 = time.time()
            solve_result_h2 = heuristic.solve_greedy_jp(processed_h, num_robot)
            tok2 = time.time()
            greedy_h2 = tok2 -tik2

            tik3 = time.time()
            solve_result_h3 = heuristic.solve_greedy_jp_compare_start_time(processed_h, num_robot)
            tok3 = time.time()
            greedy_jp_time_2 = tok3 -tik3

            depot = processed_h.depot
            tasks = processed_h.tasks
            c_travel = helpers.create_task_dist_dict(depot,tasks)
            c_operate = helpers.create_operate_dict(depot,tasks)
            tik = time.time()
            solve_opt = heuristic.TwoOptSolver(processed_h,c_travel, c_operate)
            graph,makespan_2opt = solve_opt.solve(total_robots= num_robot)
            tok = time.time()
            two_opt_time = tok - tik

            solve_log = []
            solve_log.append(data_path)
            solve_log.append(num_robot)
            solve_log.append(solve_result_h2.get_makespan().round(2))
            solve_log.append(np.round(greedy_h2,2))
            solve_log.append(solve_result_h3.get_makespan().round(2))
            solve_log.append(np.round(greedy_jp_time_2,2))
            solve_log.append(makespan_2opt.round(2))
            solve_log.append(np.round(two_opt_time,2))
            now = datetime.datetime.now()
            solve_log.append(now.strftime("%Y-%m-%d %H:%M:%S"))
            with open('solve_log_h_2opt.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)

In [3]:
first_names = ['lc','lr','lrc']
sec_names = [i for i in range(101,115)] +[i for i in range(201,215)]
# third_names = [i for i in range(1,11)]
data_paths =[f'lim/100/{i}{j}-5-uniform.csv'for i in first_names for j in sec_names]#
num_robots = [10*i for i in range(1,2)]
file_type = 'csv'
is_depot_task = False

In [8]:
## Mass solve Viet 

import logging
logging.basicConfig(format=cfg.LOG_FORMATTER, level=cfg.LOG_LEVEL, datefmt='%d-%b-%y %H:%M:%S')
consider_process_time =True
for data_path in data_paths:
    for num_robot in num_robots:
        try:
            print(f'Solving for {data_path} with {num_robot} robots')
            raw = get_data(data_path, file_type, is_depot_task)
            processed_h = heuristic.process_greedy_1(raw)
            depot = processed_h.depot
            tasks = processed_h.tasks
            c_travel = helpers.create_task_dist_dict(depot,tasks)
            c_operate = helpers.create_operate_dict(depot,tasks)
            tik = time.time()
            solve_viet = heuristic.VietSolverOrig(processed_h, c_travel, c_operate)
            solve_viet.solve(num_robot,consider_process_time)
            tok = time.time()
            time_viet = tok - tik
            print(f'makespan: {solve_viet.get_makespan()}')
            print('Done')

            solve_log = []
            solve_log.append(data_path)
            solve_log.append(num_robot)
            solve_log.append(solve_viet.get_makespan().round(2))
            solve_log.append(np.round(time_viet,2))
            with open('solve_log_h_Viet_09-04-process-time.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)


Solving for lim/100/lc101-5-uniform.csv with 10 robots
R_A = 163 > total_robots = 10, solving second case
C:\Users\CTNGUY~1\AppData\Local\Temp\Viet_case_2.lp
R_A = 153 > total_robots = 10, solving second case
C:\Users\CTNGUY~1\AppData\Local\Temp\Viet_case_2.lp
R_A = 143 > total_robots = 10, solving second case
C:\Users\CTNGUY~1\AppData\Local\Temp\Viet_case_2.lp
R_A = 133 > total_robots = 10, solving second case
C:\Users\CTNGUY~1\AppData\Local\Temp\Viet_case_2.lp
R_A = 123 > total_robots = 10, solving second case
C:\Users\CTNGUY~1\AppData\Local\Temp\Viet_case_2.lp
R_A = 113 > total_robots = 10, solving second case
C:\Users\CTNGUY~1\AppData\Local\Temp\Viet_case_2.lp
R_A = 103 > total_robots = 10, solving second case
C:\Users\CTNGUY~1\AppData\Local\Temp\Viet_case_2.lp
R_A = 93 > total_robots = 10, solving second case
C:\Users\CTNGUY~1\AppData\Local\Temp\Viet_case_2.lp
R_A = 83 > total_robots = 10, solving second case
C:\Users\CTNGUY~1\AppData\Local\Temp\Viet_case_2.lp
R_A = 73 > total_rob

In [14]:
## Mass solve Viet k tasks

import logging
logging.basicConfig(format=cfg.LOG_FORMATTER, level=cfg.LOG_LEVEL, datefmt='%d-%b-%y %H:%M:%S')

tasks_each_iters = [5]
consider_process_time =False
for task_each_ite in tasks_each_iters:
    for data_path in data_paths:
        for num_robot in num_robots:
            try:
                print(f'Solving for {data_path} with {num_robot} robots')
                raw = get_data(data_path, file_type, is_depot_task)
                processed_h = heuristic.process_greedy_1(raw)
                depot = processed_h.depot
                tasks = processed_h.tasks
                c_travel = helpers.create_task_dist_dict(depot,tasks)
                c_operate = helpers.create_operate_dict(depot,tasks)
                tik = time.time()
                solve_viet = heuristic.VGeneralizedSolver(processed_h, c_travel, c_operate)
                sol = solve_viet.solve(num_robot,task_each_ite,consider_process_time)
                tok = time.time()
                time_viet = tok - tik
                makespan = sol.get_makespan()
                print(f'makespan: {makespan}')
                print('Done')

                solve_log = []
                solve_log.append(data_path)
                solve_log.append(num_robot)
                solve_log.append(task_each_ite)
                solve_log.append(makespan.round(2))
                solve_log.append(np.round(time_viet,2))
                with open('solve_log_h_Viet k task large instances 23-04.csv', 'a') as f:
                    f.write(','.join([str(i) for i in solve_log])+'\n')
            except Exception as e:
                print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
                print(e)


Solving for lim/200/LC1_2_1-5-uniform.csv with 10 robots
Viet generalized sol is 1185.73
makespan: 1185.7308396855422
Done
Solving for lim/200/LC1_2_1-5-uniform.csv with 20 robots
Viet generalized sol is 616.46
makespan: 616.464354440489
Done
Solving for lim/200/LC1_2_1-5-uniform.csv with 30 robots
Viet generalized sol is 442.09
makespan: 442.08881408607806
Done
Solving for lim/200/LC1_2_2-5-uniform.csv with 10 robots
Viet generalized sol is 1333.44
makespan: 1333.4397653376848
Done
Solving for lim/200/LC1_2_2-5-uniform.csv with 20 robots
Viet generalized sol is 720.58
makespan: 720.5757736762516
Done
Solving for lim/200/LC1_2_2-5-uniform.csv with 30 robots
Viet generalized sol is 489.74
makespan: 489.74244307055653
Done
Solving for lim/200/LC1_2_3-5-uniform.csv with 10 robots
Viet generalized sol is 1736.97
makespan: 1736.9651267594204
Done
Solving for lim/200/LC1_2_3-5-uniform.csv with 20 robots
Viet generalized sol is 871.3
makespan: 871.2967926705874
Done
Solving for lim/200/LC1_2_

In [ ]:
raw = get_data(data_path, file_type, is_depot_task)
processed_h = heuristic.process_greedy_1(raw)
depot = processed_h.depot
tasks = processed_h.tasks
c_travel = helpers.create_task_dist_dict(depot,tasks)
c_operate = helpers.create_operate_dict(depot,tasks)
tik = time.time()
solve_opt = heuristic.TwoOptSolver(processed_h,c_travel, c_operate)

In [ ]:
#Get data

#Read file solve_log.csv, save it as a dataframe
import pandas as pd
solve_cplex_df = pd.read_csv('solve_log.csv', header=None)
solve_cplex_df.columns = ['data_path', 'num_robot', 'solve_time', 'objective_value', 'processed_nodes', 'gap', 'solve_date']

solve_h_df = pd.read_csv('solve_log_h.csv', header=None)
solve_h_df.columns = ['data_path', 'objective_value_1', 'time_greedy_1', 'objective_value_jp', 'time_greedy_jp']

In [ ]:
# right join 2 dataframes on column data_path
merged_df = pd.merge(solve_cplex_df, solve_h_df, on='data_path', how='right')

In [ ]:
merged_df['diff_h_1'] = np.round(1- merged_df['objective_value_1']/merged_df['objective_value'],2)
merged_df['diff_h_jp'] = np.round(1- merged_df['objective_value_jp']/merged_df['objective_value'],2)
#change order of columns in meged_df as follow: data_path, num_robot, objective_value, solve_time, objective_value_1, time_greedy_1, diff_h_1, objective_value_jp, time_greedy_jp, diff_h_jp,
merged_df = merged_df[['data_path', 'num_robot', 'objective_value', 'solve_time', 'objective_value_1', 'time_greedy_1', 'diff_h_1', 'objective_value_jp', 'time_greedy_jp', 'diff_h_jp']]
merged_df
#Remove the 'lim' prefix from data_path
merged_df['data_path'] = merged_df['data_path'].str.replace('lim/','')
#round solve_time to 2 decimal places
merged_df['solve_time'] = np.round(merged_df['solve_time'],2)
#round objective_value to 2 decimal places
merged_df['objective_value'] = np.round(merged_df['objective_value'],2)

In [ ]:
merged_df.to_latex('merged_df.tex', index=True)

In [ ]:
#Temp
import pandas as pd
solve_h_df = pd.read_csv('solve_log_h_2.csv', header=None)
solve_h_df.columns = ['data_path', 'objective_value_1', 'time_greedy_1', 'objective_value_2', 'time_greedy_2', 'objective_value_3', 'time_greedy_3']
#Keep the following columns: data_path, objective_value_2, time_greedy_2
solve_h_df = solve_h_df[['data_path', 'objective_value_2', 'time_greedy_2']]

In [ ]:
solve_h_df_2opt = pd.read_csv('solve_log_h_2opt.csv', header=None)
solve_h_df_2opt.columns = ['data_path', 'num_robot', 'objective_value_3', 'time_greedy_3', 'objective_value_2opt', 'time_2opt']

In [ ]:
#Join 2 dataframes on data_path
merged_df_2 = pd.merge(solve_h_df, solve_h_df_2opt, on='data_path', how='right')
#Change the order of columns as follow: data_path, num_robot, objective_value_2, time_greedy_2, objective_value_3, time_greedy_3, objective_value_2opt, time_2opt
merged_df_2 = merged_df_2[['data_path', 'num_robot', 'objective_value_2', 'time_greedy_2', 'objective_value_3', 'time_greedy_3', 'objective_value_2opt', 'time_2opt']]

In [ ]:
#save the data as a latex table
merged_df_2.to_latex('merged_df_2.tex', index=True)